In [1]:
from dotenv import load_dotenv
load_dotenv()

from unsloth import is_bf16_supported, FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

import json
import os
from omegaconf import OmegaConf
from PIL import Image
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import time
from trl import SFTTrainer, SFTConfig
from tqdm.notebook import tqdm
import wandb

import torch
import gc

from beyond_hate.train.utils import extract_label, convert_to_conversation_inference, resize_and_pad

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
cfg = OmegaConf.load("../../config/default.yaml")
eval_cfg = OmegaConf.load("../../config/eval.yaml")
# Merge training part with evaluation configuration
config = OmegaConf.merge(cfg.training, eval_cfg) 

In [3]:
# Load labeled data
with open(f'{cfg.out.path}/labels.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]
# Get relative paths for images
data = [{**item, 'img': '/'.join(item['img'].split('/')[-2:])} for item in data]
# Just keep items with valid image paths
data = [item for item in data if os.path.exists(f"{cfg.data.paths.hf}/{item['img']}")]

In [4]:
model, tokenizer = FastVisionModel.from_pretrained(
    config.model,
    load_in_4bit = config.load_in_4bit,
    use_gradient_checkpointing = config.use_gradient_checkpointing, 
    max_seq_length = config.max_seq_length
)

==((====))==  Unsloth 2025.6.4: Fast Llava patching. Transformers: 4.52.4.
   \\   /|    NVIDIA RTX 4000 Ada Generation. Num GPUs = 1. Max memory: 19.575 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


processor_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

In [5]:
SYSTEM_TEXT = ("You are a content moderation assistant. Aid me to"
                " label images with text as hateful or neutral."
                " Hateful image are defined as containing a direct or indirect"
                " attack on people based on characteristics, including"
                " ethnicity, race, nationality, immigration status, religion,"
                " caste, sex, gender identity, sexual orientation, and"
                " disability or disease.")

USER_TEXT = (' Considering the image and its text: "{}".'
             ' Is the content of the image and its text hateful or neutral? '
             ' Respond only with the word "Hateful" or "Neutral".')

In [8]:
FastVisionModel.for_inference(model)

results = []
for sample in tqdm(data):
    label = sample['label_hateful']
    text = sample['text']
    
    # Resize and pad the image if specified in the config
    img_size = config.image_size
    image = resize_and_pad(Image.open(f"{cfg.data.paths.hf}/{sample['img']}"), target_size=(img_size, img_size), color=(255, 255, 255))
    
    conversation = convert_to_conversation_inference(text, SYSTEM_TEXT, USER_TEXT)
    prompt = tokenizer.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = tokenizer(images=image, text=prompt, return_tensors="pt").to("cuda:0")

    # autoregressively complete prompt
    max_new_tokens = 50
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    output = tokenizer.decode(output[0], skip_special_tokens=True)
    output = output.split('[/INST]')[-1]

    results.append(
        {
            'id': sample['id'],
            'label': label,
            'output': output,
        }
    )

  0%|          | 0/343 [00:00<?, ?it/s]

You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


In [9]:
# Relabel
possible_labels = {'Hateful': 1, 'Neutral': 0}
y_true = [r['label'] for r in results]
#y_true = [1 if pred == 'Hateful' else 0 for pred in y_true]
y_pred = [extract_label(r['output'], possible_labels) for r in results]

# Filter out invalid predictions (-1)
valid = [i for i, pred in enumerate(y_pred) if pred != -1]
y_true_valid = [y_true[i] for i in valid]
y_pred_valid = [y_pred[i] for i in valid]
print(f'Invalid predictions: {len(y_pred) - len(y_true_valid)}')

accuracy = accuracy_score(y_true_valid, y_pred_valid)
precision = precision_score(y_true_valid, y_pred_valid, average='weighted')  # or 'macro' depending on your needs
recall = recall_score(y_true_valid, y_pred_valid, average='weighted')
f1 = f1_score(y_true_valid, y_pred_valid, average='weighted')
cm = confusion_matrix(y_true_valid, y_pred_valid, normalize='true')

Invalid predictions: 0


In [23]:
# Create a DataFrame for easier analysis
import pandas as pd

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Convert data to DataFrame and merge on 'id'
data_df = pd.DataFrame(data)
merged_df = results_df.merge(data_df[['id', 'label_intolerance', 'label_incivility']], on='id', how='left')

# Binarize incivility and intolerance labels (0 for 0, 1 for >0)
merged_df['label_incivility'] = (merged_df['label_incivility'] > 0).astype(int)
merged_df['label_intolerance'] = (merged_df['label_intolerance'] > 0).astype(int)

# Add predicted labels and classification correctness
merged_df['predicted'] = [extract_label(output, possible_labels) for output in merged_df['output']]
merged_df['correct'] = merged_df['label'] == merged_df['predicted']

# Filter out invalid predictions
valid_predictions = merged_df['predicted'] != -1
merged_df = merged_df[valid_predictions]

print(f"Total valid predictions: {len(merged_df)}")
print(f"Correct predictions: {merged_df['correct'].sum()}")
print(f"Wrong predictions: {(~merged_df['correct']).sum()}")

# Analyze distribution of incivility and intolerance labels in wrongly classified items
wrong_predictions = merged_df[~merged_df['correct']]

print("=== WRONGLY CLASSIFIED ITEMS ANALYSIS ===")
print(f"Total wrongly classified items: {len(wrong_predictions)}")
print()

# Distribution of incivility labels in wrong predictions
print("Incivility Label Distribution (Wrong Predictions):")
incivility_dist = wrong_predictions['label_incivility'].value_counts().sort_index()
print(f"Incivility percentage: {incivility_dist / len(wrong_predictions) * 100}")
print()

# Distribution of intolerance labels in wrong predictions
print("Intolerance Label Distribution (Wrong Predictions):")
intolerance_dist = wrong_predictions['label_intolerance'].value_counts().sort_index()
print(f"Intolerance percentage: {intolerance_dist / len(wrong_predictions) * 100}")
print()

# Analyze specific patterns in wrong predictions
print("=== DETAILED ANALYSIS OF WRONG PREDICTIONS ===")
print()

# False Positives (predicted hateful, actually neutral)
false_positives = wrong_predictions[(wrong_predictions['label'] == 0) & (wrong_predictions['predicted'] == 1)]
print(f"False Positives (predicted hateful, actually neutral): {len(false_positives)}")
if len(false_positives) > 0:
    print("Incivility in False Positives:")
    print(false_positives['label_incivility'].value_counts().sort_index())
    print("Intolerance in False Positives:")
    print(false_positives['label_intolerance'].value_counts().sort_index())
print()

# False Negatives (predicted neutral, actually hateful)
false_negatives = wrong_predictions[(wrong_predictions['label'] == 1) & (wrong_predictions['predicted'] == 0)]
print(f"False Negatives (predicted neutral, actually hateful): {len(false_negatives)}")
if len(false_negatives) > 0:
    print("Incivility in False Negatives:")
    print(false_negatives['label_incivility'].value_counts().sort_index())
    print("Intolerance in False Negatives:")
    print(false_negatives['label_intolerance'].value_counts().sort_index())

Total valid predictions: 347
Correct predictions: 310
Wrong predictions: 37
=== WRONGLY CLASSIFIED ITEMS ANALYSIS ===
Total wrongly classified items: 37

Incivility Label Distribution (Wrong Predictions):
Incivility percentage: label_incivility
0    54.054054
1    45.945946
Name: count, dtype: float64

Intolerance Label Distribution (Wrong Predictions):
Intolerance percentage: label_intolerance
0    48.648649
1    51.351351
Name: count, dtype: float64

=== DETAILED ANALYSIS OF WRONG PREDICTIONS ===

False Positives (predicted hateful, actually neutral): 20
Incivility in False Positives:
label_incivility
0    12
1     8
Name: count, dtype: int64
Intolerance in False Positives:
label_intolerance
0    15
1     5
Name: count, dtype: int64

False Negatives (predicted neutral, actually hateful): 17
Incivility in False Negatives:
label_incivility
0    8
1    9
Name: count, dtype: int64
Intolerance in False Negatives:
label_intolerance
0     3
1    14
Name: count, dtype: int64


Total valid predictions: 347
Correct predictions: 310
Wrong predictions: 37
=== WRONGLY CLASSIFIED ITEMS ANALYSIS ===
Total wrongly classified items: 37

Incivility Label Distribution (Wrong Predictions):
Incivility percentage: label_incivility
0    54.054054
1    45.945946
Name: count, dtype: float64

Intolerance Label Distribution (Wrong Predictions):
Intolerance percentage: label_intolerance
0    48.648649
1    51.351351
Name: count, dtype: float64

=== DETAILED ANALYSIS OF WRONG PREDICTIONS ===

False Positives (predicted hateful, actually neutral): 20
Incivility in False Positives:
label_incivility
0    12
1     8
Name: count, dtype: int64
Intolerance in False Positives:
label_intolerance
0    15
1     5
Name: count, dtype: int64

False Negatives (predicted neutral, actually hateful): 17
Incivility in False Negatives:
label_incivility
0    8
1    9
Name: count, dtype: int64
Intolerance in False Negatives:
label_intolerance
0     3
1    14
Name: count, dtype: int64